# RAG Fundamentals — a real retrieve-then-generate pipeline, end to end

> **This notebook is the lesson.** We build a *working* Retrieval-Augmented Generation system out of the
> exact components production RAG uses — a **real Hugging Face Wikipedia corpus**, a **real
> sentence-transformers embedder**, a **real FAISS vector index**, a **real cross-encoder reranker**, and a
> **real LLM** through the Hugging Face Inference API. No toy corpus, no hand-rolled math stand-ins, no
> stubbed generation. Every number and every answer below is produced live by running the cell.

You could screen-share this and teach RAG from it: each step is preceded by a plain-English *why*, then the
real code, then the real output. By the end you will have watched a language model **hallucinate a confident
wrong answer from memory**, then **get it right — with a citation — once retrieval put the right Wikipedia
passage in front of it.** That contrast, on real data, is the whole idea of RAG.

**What we build, in order**

1. Load a **real corpus** (3,200 Wikipedia passages + 918 QA pairs) and look at it.
2. **Embed** every passage with a real bi-encoder → a 3,200 × 384 matrix.
3. Build a **real FAISS index** and **retrieve** the nearest passages for a question — with real cosine scores and latency.
4. **Augment + generate**: the headline **hallucination-vs-grounding** contrast on a real obscure fact.
5. Show the **honest "I don't know"** when the answer isn't in the corpus.
6. **Rerank** with a cross-encoder and watch it *fix a real retrieval miss*.
7. **Measure recall@k** on 200 real QA pairs — dense vs reranked — the number that governs RAG quality.

**Reproducibility & honesty.** Retrieval is deterministic (fixed weights + exact FAISS). Generation calls a
*hosted* LLM at `temperature=0` for near-determinism, but a hosted endpoint can still vary run to run
(provider routing, model updates). We never fake an answer — if the API is down, the call raises. The banner
in Step 0 prints the exact models, dataset, and library versions so you can see precisely what produced each
result.

## Step 0 — Setup: the libomp guard, then the real components

Before anything else, one environment quirk that will bite you on macOS: **FAISS and PyTorch each bundle their
own OpenMP runtime**, and when both load, they collide and *segfault* the whole process. The fix is to set two
environment variables **before importing faiss or torch**, so we do it in the very first cell. `rag_fundamentals`
also sets them at its top — belt and suspenders.

Everything real lives in the companion module **`rag_fundamentals.py`** (a typed, ~250-line production-shaped
mini-RAG). We import it and let it print a banner of exactly what is in play.

In [1]:
# --- OpenMP guard: MUST run before faiss / torch import (they each load libomp -> segfault on macOS)
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # tolerate the two libomp copies
os.environ.setdefault("OMP_NUM_THREADS", "1")            # pin threads so faiss + torch don't fight
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false") # silence the HF tokenizers fork warning
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

# The real pipeline. Importing does NOT hit the network or load models — construction does.
from rag_fundamentals import RagPipeline, load_corpus, TOP_K, RERANK_CANDIDATES

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in your environment — the corpus + LLM come from Hugging Face."
print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))
print("OpenMP guard set:", os.environ["KMP_DUPLICATE_LIB_OK"], "| threads:", os.environ["OMP_NUM_THREADS"])

HF_TOKEN present: True
OpenMP guard set: TRUE | threads: 1


## Step 1 — Load a real corpus and *look* at it

RAG can only surface knowledge that **exists in the corpus** — so the corpus is not a detail, it *is* the
knowledge base. We use [`rag-datasets/rag-mini-wikipedia`](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia):
real Wikipedia passages plus real question/answer pairs, small enough to embed in seconds but large enough
(3,200 passages) that retrieval has to *actually choose* — nearest-neighbour search over thousands of vectors,
not a hand-picked handful.

Before trusting any pipeline, always **look at the raw data**: how many passages, how long are they, what does
one actually say, and what do the questions look like? A five-minute look here saves hours of debugging a
retriever that's quietly matching on the wrong thing.

In [2]:
passages, qa = load_corpus()   # real download from the HF Hub (cached after the first run)

import numpy as np
word_lengths = np.array([len(p.split()) for p in passages])
print(f"passages : {len(passages)}")
print(f"QA pairs : {len(qa)}")
print(f"passage length (words) — min {word_lengths.min()}, "
      f"median {int(np.median(word_lengths))}, mean {word_lengths.mean():.1f}, max {word_lengths.max()}")
print("\n--- a real passage (doc[0]) ---")
print(passages[0])
print("\n--- three real question/answer pairs ---")
for row in (qa[0], qa[7], qa[55]):
    print(f"Q: {row['question']}\n   A: {row['answer']}")

passages : 3200
QA pairs : 918
passage length (words) — min 1, median 48, mean 62.1, max 425

--- a real passage (doc[0]) ---
Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.

--- three real question/answer pairs ---
Q: Was Abraham Lincoln the sixteenth President of the United States?
   A: yes
Q: When did the Gettysburg address argue that America was born?
   A: 1776
Q: What happened in 1745?
   A: The scale was reversed


Notice the passages are already reasonable **chunks** — a few dozen words each, one topic apiece. In a real
system *you* would produce these by splitting your own documents (that decision is consequential enough to be
the next chapter, *Document Chunking Strategies*); this dataset ships pre-chunked so we can focus on retrieve →
generate. The questions are genuine factoids ("When did the Gettysburg address argue America was born?"),
exactly the kind of knowledge a frozen model either has, half-has, or confidently invents.

## Step 2 — Embed the corpus and build a real vector index

Now the machine that makes retrieval match on **meaning**, not just keywords. A **bi-encoder** embedding model
maps each passage to a dense vector so that *semantically similar text lands at nearby points* — "liftoff date"
can find a passage that says "was launched on," even with zero shared words.

Two design choices that matter, both handled for us by `RagPipeline.__init__`:

- **Normalize to unit vectors.** With every vector L2-normalized, **cosine similarity collapses to a plain dot
  product** — so FAISS's inner-product index (`IndexFlatIP`) computes exact cosine, and the whole corpus is
  scored in one matrix multiply.
- **Same model for corpus and queries.** Embed passages with model A and queries with model B and the two
  vector spaces don't align — similarities become meaningless. This is the single most common *silent* RAG bug.

Constructing the pipeline embeds all 3,200 passages (a few seconds on CPU/MPS) and adds them to FAISS.

In [3]:
pipeline = RagPipeline(passages)   # embeds the corpus + builds the FAISS index

print("embeddings shape (n_passages, dim):", pipeline.embeddings.shape)
print("dtype:", pipeline.embeddings.dtype, "| embed time: %.1fs" % pipeline.embed_seconds)
print("every row unit-norm:", np.allclose(np.linalg.norm(pipeline.embeddings, axis=1), 1.0, atol=1e-5))
print("FAISS index type:", type(pipeline.index).__name__, "| vectors indexed:", pipeline.index.ntotal)

embeddings shape (n_passages, dim): (3200, 384)
dtype: float32 | embed time: 9.5s
every row unit-norm: True
FAISS index type: IndexFlatIP | vectors indexed: 3200


In [4]:
# The reproducibility banner: exactly what is producing every result below.
from rag_fundamentals import _print_banner
_print_banner(pipeline)

REAL mini-RAG -- reproducibility banner
  numpy 2.4.6 | faiss 1.14.3 | torch 2.12.0 | device mps
  datasets 5.0.0 | sentence-transformers 5.6.0 | hub 1.18.0
  dataset : rag-datasets/rag-mini-wikipedia  (3200 passages)
  embed   : sentence-transformers/all-MiniLM-L6-v2  (384-d, 9.5s to embed corpus)
  rerank  : cross-encoder/ms-marco-MiniLM-L-6-v2
  generate: meta-llama/Llama-3.1-8B-Instruct -> meta-llama/Llama-3.3-70B-Instruct -> deepseek-ai/DeepSeek-V3 (first that responds)


## Step 3 — Retrieve: nearest-neighbour search, with real scores and latency

Retrieval is the heart of RAG, and it is startlingly simple: **embed the question with the same model, then ask
FAISS for the passages whose vectors are closest** (highest cosine). "Find the answer" is literally "find the
nearest points."

Our headline question is chosen deliberately: *"What was reversed about the temperature scale in 1745?"* The
answer (the Celsius scale was flipped — by **Linnaeus**, not Celsius) is a genuinely obscure historical detail.
Watch which real Wikipedia passages come back, their real cosine scores, and how fast the search is over 3,200
vectors.

In [5]:
question = "What was reversed about the temperature scale in 1745?"

import time
t = time.perf_counter()
hits = pipeline.retrieve(question, k=5)
latency_ms = (time.perf_counter() - t) * 1000

print(f'QUERY: "{question}"')
print(f"FAISS search over {pipeline.index.ntotal} vectors took {latency_ms:.1f} ms\n")
for rank, h in enumerate(hits, 1):
    print(f"{rank}. doc[{h.doc_id}]  cos={h.score:.3f}\n   {h.text[:160]}{'...' if len(h.text) > 160 else ''}\n")

QUERY: "What was reversed about the temperature scale in 1745?"
FAISS search over 3200 vectors took 1544.6 ms

1. doc[144]  cos=0.539
   Celsius founded the Uppsala Astronomical Observatory in 1741, and in 1742 he proposed the Celsius temperature scale in a paper to the Royal Swedish Academy of S...

2. doc[145]  cos=0.530
   Anders Celsius was the first to perform and publish careful experiments aiming at the definition of an international temperature scale on scientific grounds. In...

3. doc[140]  cos=0.420
   The observatory of Anders Celsius, from a contemporary engraving.

4. doc[237]  cos=0.374
   Pascal replicated the experiment in Paris by carrying a barometer up to the top of the bell tower at the church of Saint-Jacques-de-la-Boucherie, a height of ab...

5. doc[139]  cos=0.368
   Anders Celsius



Two things to read off those results. First, **search is ~milliseconds** even over thousands of vectors —
retrieval is cheap; the LLM call will dominate latency. Second, the top passage **doc[144]** is the one that
actually contains the answer — read its full text below and you'll find the exact fact the model needs. The
retriever surfaced the right page; now the generator has to read it.

In [6]:
print("Full text of the top retrieved passage doc[%d]:\n" % hits[0].doc_id)
print(hits[0].text)

Full text of the top retrieved passage doc[144]:

Celsius founded the Uppsala Astronomical Observatory in 1741, and in 1742 he proposed the Celsius temperature scale in a paper to the Royal Swedish Academy of Sciences. His thermometer had 100 for the freezing point of water and 0 for the boiling point. The scale was reversed by Carolus Linnaeus in 1745, to how it is today  Linnaeus' thermometer .


## Step 4 — The headline contrast: hallucination (closed book) vs grounding (open book)

This is the cell that makes the case for RAG. We ask the **same real LLM** the **same question two ways**:

- **Ungrounded** — the bare question, answered from the model's frozen parametric memory (closed-book exam).
- **Grounded** — the question wrapped in an augmented prompt containing the retrieved passages, with the
  instruction *"answer using ONLY the context… cite the passage number"* (open-book exam).

Watch closely: the obscure "1745" fact is right on the edge of the model's memory, so the **ungrounded answer
misattributes it** — a fluent, confident, *wrong* answer (the worst failure mode, because it looks right). The
**grounded answer is correct and cites its source**, because the true fact was sitting in the prompt.

In [7]:
# UNGROUNDED: bare question, parametric memory only (a real LLM call)
ungrounded = pipeline.generate_ungrounded(question)
print("UNGROUNDED (no retrieval):")
print("  ->", ungrounded)

UNGROUNDED (no retrieval):
  -> In 1745, the temperature scale was reversed by Anders Celsius.


In [8]:
# GROUNDED: retrieve top-k -> augment -> generate (the full RAG loop)
result = pipeline.answer(question, rerank=False)   # rerank=False here to isolate plain dense RAG
print("GROUNDED (retrieve-then-generate):")
print("  ->", result.answer)
print(f"\n  model:    {result.model}")
print(f"  retrieve: {result.latency_s['retrieve_s']*1000:.1f} ms")
print(f"  generate: {result.latency_s['generate_s']:.2f} s")

GROUNDED (retrieve-then-generate):
  -> The temperature scale was reversed by Carolus Linnaeus in 1745, to how it is today [1].

  model:    meta-llama/Llama-3.1-8B-Instruct
  retrieve: 27.5 ms
  generate: 1.08 s


If everything is wired correctly you just saw the point of RAG in two answers: **the ungrounded model names
the wrong person; the grounded model names the right one and cites `[1]`.** Same model, same question — the only
difference is whether the correct passage was in the prompt. *That's* RAG: it changes **what facts the model has
access to**, not how well it reasons.

Let's look at the exact augmented prompt the grounded call sent — the "open book on the desk," made literal.

In [9]:
print(result.prompt)

Answer the question using ONLY the context below. If the context does not contain the answer, say you don't know. Cite the passage number(s) in square brackets that you used.

Context:
[1] Celsius founded the Uppsala Astronomical Observatory in 1741, and in 1742 he proposed the Celsius temperature scale in a paper to the Royal Swedish Academy of Sciences. His thermometer had 100 for the freezing point of water and 0 for the boiling point. The scale was reversed by Carolus Linnaeus in 1745, to how it is today  Linnaeus' thermometer .
[2] Anders Celsius was the first to perform and publish careful experiments aiming at the definition of an international temperature scale on scientific grounds. In his Swedish paper "Observations of two persistent degrees on a thermometer" he reports on experiments to check that the freezing point is independent of latitude (and of atmospheric pressure). He determined the dependence of the boiling of water with atmospheric pressure (in excellent agreement 

## Step 5 — The honest "I don't know" when the corpus can't help

RAG's discipline cuts both ways. If the answer **isn't in the corpus**, retrieval returns junk (low similarity),
and — because our prompt says *"if the context does not contain the answer, say you don't know"* — a
well-behaved grounded system **refuses instead of inventing**. That refusal is a *feature*: it's the difference
between a system you can trust and one that bluffs.

We ask something the 2007-era Wikipedia corpus obviously can't contain — yesterday's Nvidia stock price — and
watch the top cosine score fall to noise and the grounded answer come back honest.

In [10]:
out_of_corpus = "What was the closing share price of Nvidia stock yesterday?"
oo_hits = pipeline.retrieve(out_of_corpus, k=3)
print(f'QUERY (out of corpus): "{out_of_corpus}"\n')
print("Best retrieved passages (note the low, junk-level cosine scores):")
for rank, h in enumerate(oo_hits, 1):
    print(f"  {rank}. doc[{h.doc_id}] cos={h.score:.3f} | {h.text[:70]!r}")

oo_result = pipeline.answer(out_of_corpus, rerank=False)
print("\nGROUNDED answer:", oo_result.answer)

QUERY (out of corpus): "What was the closing share price of Nvidia stock yesterday?"

Best retrieved passages (note the low, junk-level cosine scores):
  1. doc[112] cos=0.294 | '125px'
  2. doc[2785] cos=0.280 | 'Closing ceremony for the National Stadium'
  3. doc[136] cos=0.272 | '150px'



GROUNDED answer: I don't know.


Compare the top cosine here (~0.29 — noise) with the ~0.54 the in-corpus question scored. **The similarity
score is itself a signal**: a low best-match is a strong hint that the corpus doesn't hold the answer, and many
production systems set a similarity floor below which they refuse or fall back. Retrieval didn't just fail
quietly — it *told us* it had nothing relevant.

## Step 6 — Reranking: fix a real retrieval miss with a cross-encoder

The fast bi-encoder embeds the query and each passage **independently**, so it can miss subtle relevance. A
**cross-encoder** reads the *(query, passage) pair together* and scores their relevance directly — far more
accurate, but too slow to run over the whole corpus. The production pattern: **retrieve a wide shortlist with
the fast dense stage, then rerank that shortlist with the cross-encoder.**

Here is a real miss it fixes. For *"When was Abraham Lincoln inaugurated as president?"* the bi-encoder ranks a
near-useless title chunk **"Young Abraham Lincoln"** at #1 (it's lexically Lincoln-y but says nothing about an
inauguration). The cross-encoder reads each passage against the actual question and **demotes the junk to last,
promoting the real inauguration passage to #1.**

In [11]:
rerank_q = "When was Abraham Lincoln inaugurated as president?"
dense = pipeline.retrieve(rerank_q, k=5)
reranked = pipeline.rerank(rerank_q, dense, k=5)   # cross-encoder re-scores the same 5 candidates

print("DENSE order (bi-encoder cosine):")
for rank, h in enumerate(dense, 1):
    print(f"  {rank}. doc[{h.doc_id}] cos={h.score:.3f} | {h.text[:64]!r}")

print("\nRERANKED order (cross-encoder relevance logit):")
for rank, h in enumerate(reranked, 1):
    print(f"  {rank}. doc[{h.doc_id}] ce={h.score:+.2f} | {h.text[:64]!r}")

DENSE order (bi-encoder cosine):
  1. doc[288] cos=0.654 | 'Young Abraham Lincoln'
  2. doc[322] cos=0.585 | 'Photograph showing the March 4, 1861, inauguration of Abraham Li'
  3. doc[278] cos=0.583 | 'Abraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the s'
  4. doc[357] cos=0.572 | "Lincoln's second inauguration on March 4, 1865. In the photo, Li"
  5. doc[341] cos=0.570 | 'On March 4 1865, Lincoln delivered his second inaugural address,'

RERANKED order (cross-encoder relevance logit):
  1. doc[322] ce=+9.37 | 'Photograph showing the March 4, 1861, inauguration of Abraham Li'
  2. doc[357] ce=+8.00 | "Lincoln's second inauguration on March 4, 1865. In the photo, Li"
  3. doc[341] ce=+6.58 | 'On March 4 1865, Lincoln delivered his second inaugural address,'
  4. doc[278] ce=+4.46 | 'Abraham Lincoln (February 12, 1809 â\x80\x93 April 15, 1865) was the s'
  5. doc[288] ce=-2.99 | 'Young Abraham Lincoln'


Read the flip: dense put **doc[288]** ("Young Abraham Lincoln") first with the *highest* cosine — and the
cross-encoder gives it the *lowest* score (a big negative logit), dropping it to last. The passage that actually
states the March 4, 1861 inauguration jumps to #1. **The bi-encoder retrieves; the cross-encoder judges.** This
is why serious RAG systems almost always add a reranker — it's the cheapest large win in retrieval quality,
which we'll quantify next. (Full treatment: *Re-ranking with Cross-Encoders*.)

## Step 7 — Measure what actually matters: recall@k, dense vs reranked

Everything above was one query at a time. To *engineer* a RAG system you need a number: **recall@k** — across
many real questions, how often is a supporting passage somewhere in the top-k retrieved? If the answer isn't
retrieved, a perfect LLM still can't answer it — so **recall@k is the ceiling on RAG quality.**

We measure it honestly on **real QA pairs**: for each question, a passage counts as "supporting" if its text
contains the gold answer string (we keep short, non-yes/no answers so containment is meaningful). Then we
compute recall@k for the dense retriever, and — the payoff — recall after cross-encoder reranking of the top-20.

This cell does real work (it cross-encodes hundreds of pairs), so it takes ~20–30s.

In [12]:
import re

def normalize(text: str) -> str:
    return re.sub(r"[^a-z0-9 ]", " ", text.lower()).strip()

# Build an evaluation set: real questions whose short answer string appears in >=1 passage.
eval_set = []
for row in qa:
    answer = row["answer"].strip()
    if answer.lower() in ("yes", "no") or len(answer.split()) > 6:
        continue
    norm_answer = normalize(answer)
    if len(norm_answer) < 2:
        continue
    gold = {i for i, p in enumerate(passages) if norm_answer in normalize(p)}
    if gold:
        eval_set.append((row["question"], gold))
    if len(eval_set) >= 200:
        break

print(f"evaluation questions with a locatable gold passage: {len(eval_set)}")

evaluation questions with a locatable gold passage: 200


In [13]:
# Dense recall@k: one batched search, then count hits at each k.
questions = [q for q, _ in eval_set]
q_emb = pipeline.embedder.encode(questions, normalize_embeddings=True,
                                 batch_size=128, show_progress_bar=False).astype("float32")
Ks = [1, 2, 3, 5, 10, 20]
_, ids = pipeline.index.search(q_emb, max(Ks))

dense_recall = {}
for k in Ks:
    hits = sum(len({int(x) for x in ids[row, :k]} & gold) > 0 for row, (_, gold) in enumerate(eval_set))
    dense_recall[k] = hits / len(eval_set)

print("DENSE recall@k:")
for k in Ks:
    print(f"  recall@{k:<2} = {dense_recall[k]:.3f}")

DENSE recall@k:
  recall@1  = 0.420
  recall@2  = 0.525
  recall@3  = 0.575
  recall@5  = 0.660
  recall@10 = 0.720
  recall@20 = 0.740


In [14]:
# Reranked recall: take the top-20 dense hits, cross-encoder rerank, measure recall@{1,3,5}.
from sentence_transformers import CrossEncoder
from rag_fundamentals import RERANK_MODEL_ID

reranker = CrossEncoder(RERANK_MODEL_ID)
rerank_recall = {1: 0, 3: 0, 5: 0}
for row, (q, gold) in enumerate(eval_set):
    candidate_ids = [int(x) for x in ids[row, :RERANK_CANDIDATES]]
    scores = reranker.predict([(q, passages[i]) for i in candidate_ids])
    ordered = [candidate_ids[j] for j in np.argsort(scores)[::-1]]
    for k in rerank_recall:
        rerank_recall[k] += int(len(set(ordered[:k]) & gold) > 0)
for k in rerank_recall:
    rerank_recall[k] /= len(eval_set)

print("Dense vs reranked recall (reranking the top-20 dense candidates):")
print(f"  recall@1 :  dense {dense_recall[1]:.3f}  ->  reranked {rerank_recall[1]:.3f}")
print(f"  recall@3 :  dense {dense_recall[3]:.3f}  ->  reranked {rerank_recall[3]:.3f}")
print(f"  recall@5 :  dense {dense_recall[5]:.3f}  ->  reranked {rerank_recall[5]:.3f}")

Dense vs reranked recall (reranking the top-20 dense candidates):
  recall@1 :  dense 0.420  ->  reranked 0.650
  recall@3 :  dense 0.575  ->  reranked 0.710
  recall@5 :  dense 0.660  ->  reranked 0.710


There's the engineering story in three numbers. **Dense recall@1 is modest** — the single top hit is the
supporting passage only some of the time — and it climbs as you widen k (more passages retrieved ⇒ more likely
the answer is in there, at the cost of noise and tokens). **Reranking lifts recall@1 dramatically** by promoting
the genuinely-relevant passage the bi-encoder buried: you get top-5 quality *at rank 1*, which is exactly what
you want when you can only afford to stuff a few passages into the prompt.

This is the through-line of the entire RAG domain: **the generator is the easy part — RAG is won or lost at
retrieval.** Every later chapter (better chunking, better embeddings, hybrid search, reranking, query
transformation, evaluation) is a different lever on *these* recall numbers.

## Recap — what you just ran

- **A real corpus** (3,200 Wikipedia passages) embedded by a **real bi-encoder** into a **real FAISS index** —
  retrieval is nearest-neighbour search over dense vectors, ~milliseconds per query.
- **The headline contrast**: the same real LLM **hallucinated** an obscure fact from memory, then answered it
  **correctly and with a citation** once retrieval put the right passage in the prompt. That is RAG.
- **Honest refusal**: when the answer wasn't in the corpus, low similarity → the grounded system said "I don't
  know" instead of inventing.
- **Reranking** with a real cross-encoder **fixed a real retrieval miss** and, measured over 200 real QA pairs,
  **lifted recall@1 substantially**.
- The lesson that organizes everything after this: **suspect retrieval first** — almost every RAG failure is a
  retrieval failure wearing a generation costume.

**Try it yourself:** change `question` in Step 3 to any factoid the corpus might cover (Lincoln, Uruguay,
leopards, otters, the Eiffel Tower — all real topics in this corpus), and watch the retrieve → augment →
generate loop run live. Swap the generation model in `rag_fundamentals.GEN_MODEL_IDS`, or raise `k`, and see how
the answer and latency move.

See the companion concept page **[01-RAG-Fundamentals.md](../01-RAG-Fundamentals.md)** for the intuition, the
math (cosine similarity, top-k, the RAG marginalization), the Mermaid pipeline diagram, and the four classic
failure modes — each shown failing on this same real data.